# 01 - Exploring the KFC reviews
First look at the data, the problems I found while looking, and how each one led to a change in `src/kfc_absa/cleaning.py`.

Run `python scripts/run_clean.py` first to create `data/processed/reviews.csv`. Sections 4 and 5 also read the raw file in `data/raw/`, which is not in the repo (see the README).

In [ ]:
import collections

import pandas as pd

from kfc_absa.cleaning import detail_keys, normalize_text
from kfc_absa.config import RAW_PATH
from kfc_absa.data_io import load_raw_reviews, load_reviews

pd.set_option("display.max_colwidth", 90)
df = load_reviews()

## 1. Size and columns

In [ ]:
print(df.shape)
list(df.columns)

In [ ]:
df[["id", "rating", "iso_date", "clean_text"]].sample(10, random_state=42)

## 2. Ratings and missing values
Ratings are heavily skewed to 1 star. The aspect scores (`food`, `service`, `atmosphere`) and the details (`wait_time`, `order_type`...) are only filled when the reviewer chose to give them.

In [ ]:
df["rating"].value_counts().sort_index()

In [ ]:
df.isna().sum().sort_values(ascending=False)

## 3. Review length

In [ ]:
df["clean_text"].str.len().describe()

## 4. Problem found: `order_type` held the wrong data
The first version of the cleaning code filled `order_type` from the raw field `meal_type`, so Dine in / Take out / Delivery never reached the table. The raw file has both fields:

In [ ]:
if RAW_PATH.exists():
    raw = load_raw_reviews()
    print(f"{len(detail_keys(raw))} different keys inside 'details' across all reviews")
    for key in ["order_type", "meal_type"]:
        counts = collections.Counter((r.get("details") or {}).get(key) for r in raw)
        print(f"\n{key}:", dict(counts.most_common()))
else:
    print("Raw file not found, skipping (run scripts/run_scrape.py, or copy raw_reviews.json into data/raw/).")

Fixed in `flatten_reviews`: `order_type` and `meal_type` are now separate columns. The table agrees with the raw counts:

In [ ]:
print(df["order_type"].value_counts(dropna=False).to_dict())
print(df["meal_type"].value_counts(dropna=False).to_dict())

## 5. Problem found: messy text
Reviews contain line breaks, curly quotes and extra spaces. `normalize_text` adds a `clean_text` column and keeps the original `text`.
How many reviews changed, and what does a change look like?

In [ ]:
changed = df[df["text"] != df["clean_text"]]
print(f"{len(changed)} of {len(df)} reviews changed")
changed[["id", "text", "clean_text"]].head(3)

The function applied to the original text reproduces the saved column exactly, so the table and the code agree:

In [ ]:
(df["text"].map(normalize_text) == df["clean_text"]).all()

## 6. Hidden characters
`print` and tables hide characters you cannot see. `isprintable()` finds them, `Counter` counts them.

In [ ]:
found = collections.Counter(ch for text in df["text"] for ch in text if not ch.isprintable())
for ch, n in found.most_common():
    print(repr(ch), hex(ord(ch)), n)

Two kinds: the line break (handled by the whitespace rule) and one zero-width joiner (`U+200D`). That joiner is part of an emoji, so removing it splits the emoji apart.

**Decision:** do not list unwanted characters one by one. That only covers what I happened to see and fails on new data. `normalize_text` removes invisible characters by Unicode category (`Cf` format, `Cc` control) and keeps a small configurable set (`keep`) of joiners. See `tests/test_cleaning.py`.

## 7. Problem found: a fixed list of detail fields dropped data
The first version of `flatten_reviews` copied a fixed list of 7 fields from `details` (`order_type`, `meal_type`, `price_per_person`, `food`, `service`, `atmosphere`, `wait_time`). I chose them by looking at a few reviews. Compare with every key that exists in the raw data:

In [ ]:
if RAW_PATH.exists():
    raw = load_raw_reviews()
    all_keys = detail_keys(raw)
    first_seven = {"order_type", "meal_type", "price_per_person", "food", "service", "atmosphere", "wait_time"}
    print(f"{len(all_keys)} detail fields exist; the fixed list covered {len(first_seven & set(all_keys))}")
    print("Dropped:", [k for k in all_keys if k not in first_seven])
else:
    print("Raw file not found, skipping.")

A list typed by hand misses whatever I did not look at, and another business has different fields (a hotel has no `wait_time`). **Decision:** `flatten_reviews` now creates a column for every field found in the data. Pass `detail_fields=[...]` to choose columns yourself. How filled is each one?

In [ ]:
detail_cols = [c for c in df.columns if c in set(detail_keys(load_raw_reviews()))] if RAW_PATH.exists() else []
df[detail_cols].notna().sum().sort_values(ascending=False)

## 8. Checking text quality without reading it
Printing reviews to find problems does not work on private data. `profile_texts` returns only counts, so the result can be shared safely. The `characters_by_unicode_category` entry shows why the rules above are by category: `Cc` is mostly line breaks, `Cf` is the kept joiner.

In [ ]:
from kfc_absa.profile import profile_texts

profile_texts(df["text"])

## Next
- Language detection
- Aspect-based sentiment analysis